# 02 · Sync Measurements to Lakebase (Postgres)

**Lakebase project:** `projects/oktex/branches/production/endpoints/primary` (Autoscaling tier)  
**Database:** `oktex`

The OkTex measurement tables are synced from the local generator (which mirrors the Delta tables built in notebook 01) into Lakebase Postgres. The Databricks App reads these tables **live** at request time. Verification queries below run over `psql` against the live endpoint using a short-lived OAuth credential.

### 1. Run the Delta → Lakebase sync and verify (live)

In [ ]:
import sync_lakebase
sync_lakebase.main()

# Lakebase sync -> ep-ancient-recipe-d2pmfxiq.database.us-east-1.cloud.databricks.com  db=oktex

## Step 1: create database
database 'oktex' ready

## Step 2: create tables and load rows
tables created and loaded

## Step 3: verification (live psql against Lakebase)
### row counts
table_name        | rows 
-------------------------+------
 dim_meters              |   43
 fact_daily_measurements | 2580
 pipeline_segments       |   25
(3 rows)

### date coverage
first_day  |  last_day  | days 
------------+------------+------
 2026-06-30 | 2026-08-28 |   60
(1 row)

### today's flow by meter type
meter_type   | meters | total_actual_dth 
---------------+--------+------------------
 BIDIRECTIONAL |      8 |           195753
 DELIVERY      |     27 |           341154
 RECEIPT       |      8 |           536264
(3 rows)

### sample: today's top 8 meters
meter_id  |       meter_name       | meter_type | actual_dth | pressure_psig | variance_pct 
------------+------------------------+---------

### 2. Confirm the app's read view exists and returns today's rows

In [ ]:
host, token, email = sync_lakebase.conn_info()
r = sync_lakebase.psql(host, token, email, 'oktex',
    "SELECT meter_id, meter_name, meter_type, actual_dth, pressure_psig "
    "FROM v_latest_measurements ORDER BY actual_dth DESC LIMIT 5;")
print(r.stdout)

  meter_id  |       meter_name       | meter_type | actual_dth | pressure_psig 
------------+------------------------+------------+------------+---------------
 OKT-DN1-01 | Del Norte 1            | RECEIPT    |      88718 |         840.7
 OKT-OK4-02 | OK-4 Rec               | RECEIPT    |      73248 |         732.1
 OKT-O12-03 | Mustang/Rodman Residue | RECEIPT    |      73216 |         846.9
 OKT-OK1-01 | Jackson T. Hardeman    | RECEIPT    |      71508 |         701.1
 OKT-OK9-06 | OFS Leedy Plant        | RECEIPT    |      62448 |         900.9
(5 rows)


